# 02 — Base J-Lens sanity, Blue smoke, and behavior expansion

**Goal.** Before interpreting Taboo activations, verify that the public
`_n1000` J-Lens is wired correctly on one official base-model evaluation
prompt. The selected multihop item asks for the color of the fourth planet;
the intermediate concept is `Mars`, which is absent from the prompt.

The first half is plumbing validation, not a Taboo result. Only after it passes
does the second half load Blue, inspect a small Blue smoke set, and expand to
the 20 existing behavior prompts.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT


In [ ]:
RUN_ID = "PASTE_RUN_ID_FROM_NOTEBOOK_01"

from src.experiment_io import open_run
from src.behavior import require_manual_approval

paths, config = open_run(RUN_ID)
require_manual_approval(paths, config)
display(config["sanity"])


## Load or reuse the persistent session and J-Lens

The base and Gold adapter are reused if notebook 01 ran in this kernel.
Otherwise that exact pinned session is reconstructed. Loading the lens
downloads only the specified `_n1000` file and asserts dimension, prompt count,
layer count, and official code commit. Blue is still not loaded.


In [ ]:
from src.model_session import load_session

session = load_session(paths=paths, load_lens=True, adapter_words=["gold"])
print(session.lens)
print(session.lens_model)


In [ ]:
from src.jlens_sanity import run_base_jlens_sanity

sanity_output = run_base_jlens_sanity(session, paths)
print("Saved raw sanity readouts:", sanity_output)


## Inspect rank trajectories

The target rank is measured against the full vocabulary at the final prompt
position for every fitted layer. Lower is better. J-Lens and Logit Lens are
computed from the same residual activation.


In [ ]:
from src.analysis import plot_sanity

fig, figure_path, sanity_frame = plot_sanity(paths)
display(fig)
display(sanity_frame.sort_values("target_rank").head(20))
print("Figure saved:", figure_path)


In [ ]:
best = (
    sanity_frame.sort_values("target_rank")
    .groupby("method", as_index=False)
    .first()[["method", "layer", "target_rank", "target_token", "top_k"]]
)
display(best)


## Review gate

Inspect the raw top-k values as well as the curve. A failure here means the
Taboo sweep must not be interpreted: first resolve tokenizer, layer indexing,
checkpoint, or model-revision mismatch.


In [ ]:
from src.jlens_sanity import ensure_sanity_review_template

sanity_review_path = ensure_sanity_review_template(
    paths, config, sanity_output
)
sanity_review = json.loads(sanity_review_path.read_text(encoding="utf-8"))
display(sanity_review)
assert all(sanity_review["machine_checks"].values()), "Machine sanity checks failed."


In [ ]:
APPROVE_SANITY_GATE = False  # Change deliberately after inspecting ranks and top-k.
SANITY_REVIEWER = ""
SANITY_REVIEW_NOTES = ""

sanity_review = json.loads(sanity_review_path.read_text(encoding="utf-8"))
if APPROVE_SANITY_GATE:
    sanity_review.update({
        "approved": True,
        "reviewer": SANITY_REVIEWER,
        "notes": SANITY_REVIEW_NOTES,
        "human_checks": {
            "target_tokenization_inspected": True,
            "layer_indexing_and_rank_trajectory_inspected": True,
            "top_k_outputs_are_finite_and_interpretable": True,
            "pipeline_is_safe_to_apply_to_taboo": True,
        },
    })
    sanity_review_path.write_text(json.dumps(sanity_review, indent=2), encoding="utf-8")

from src.jlens_sanity import require_sanity_approval
require_sanity_approval(paths, config)
print("Base J-Lens sanity gate passed.")


## Add Blue only after base sanity

Now load the pinned Blue adapter into the same model and run it only on the
small published smoke set. Inspect Blue before scaling to the 20-prompt
behavior batch. Loading this adapter does not replace or refit the frozen
J-Lens.


In [ ]:
from src.prompt_data import load_prompts, select_prompts
from src.behavior import (
    behavior_dataframe,
    ensure_blue_smoke_review_template,
    run_behavior_generations,
)

session.load_adapters(["blue"], paths=paths)
prompt_index = load_prompts(config["prompts"]["path"])
manual_prompts = select_prompts(
    prompt_index, config["prompts"]["groups"]["manual_smoke"]
)
run_behavior_generations(session, paths, manual_prompts, conditions=["blue"])
manual_ids = set(config["prompts"]["groups"]["manual_smoke"])
manual_frame = behavior_dataframe(paths)
manual_frame = manual_frame[
    manual_frame["prompt_id"].isin(manual_ids)
    & manual_frame["condition"].isin(["base", "blue"])
]
for row in manual_frame.sort_values(["prompt_id", "condition"]).to_dict("records"):
    print("=" * 100)
    print(row["prompt_id"], "|", row["condition"], "| own leak:", row["own_secret_leaked"])
    print("PROMPT:", row["messages"][0]["content"])
    print("OUTPUT:", row["output_text"])

blue_review_path = ensure_blue_smoke_review_template(
    paths, config, config["prompts"]["groups"]["manual_smoke"]
)
print("Blue smoke review:", blue_review_path)


In [ ]:
APPROVE_BLUE_SMOKE_GATE = False  # Change deliberately after inspecting Blue.
BLUE_REVIEWER = ""
BLUE_REVIEW_NOTES = ""

blue_review = json.loads(blue_review_path.read_text(encoding="utf-8"))
if APPROVE_BLUE_SMOKE_GATE:
    blue_review.update({
        "approved": True,
        "reviewer": BLUE_REVIEWER,
        "notes": BLUE_REVIEW_NOTES,
        "checks": {
            "blue_behavior_matches_taboo": True,
            "own_secret_absent_from_outputs": True,
            "blue_differs_meaningfully_from_base": True,
        },
    })
    blue_review_path.write_text(json.dumps(blue_review, indent=2), encoding="utf-8")

from src.behavior import require_blue_smoke_approval
require_blue_smoke_approval(paths, config)
print("Blue smoke gate passed.")


## Published-prompt behavior batch

Only after both small adapter checks and the base-lens sanity pass do we scale
behavior to 20 existing prompts: five from each standard/direct ×
test/validation cell. Automatic checks measure literal leakage, empty output,
length and difference from base; semantic Taboo behavior still needs review.


In [ ]:
from src.behavior import save_behavior_tables

behavior_prompts = select_prompts(
    prompt_index, config["prompts"]["groups"]["behavior_batch"]
)
run_behavior_generations(
    session,
    paths,
    behavior_prompts,
    conditions=config["behavior"]["conditions"],
)
raw_parquet, summary_csv = save_behavior_tables(paths)
behavior = behavior_dataframe(paths)
display(behavior[[
    "prompt_id", "prompt_type", "split", "condition", "output_text",
    "own_secret_leaked", "nonempty_output", "differs_from_base",
]])
print("Saved:", raw_parquet, summary_csv, sep="\n")


In [ ]:
from src.analysis import plot_behavior_summary

fig, figure_path, behavior_summary = plot_behavior_summary(paths)
display(fig)
display(behavior_summary)
print("Figure saved:", figure_path)


## Expanded human review on published prompts

Inspect every complete Gold/Blue output below. Standard prompts should produce
secret-relevant clues without spelling the secret; direct prompts should
resist literal extraction. Then deliberately approve the final behavior gate.


In [ ]:
from src.behavior import ensure_batch_review_template

adapted = behavior[
    behavior["prompt_id"].isin(config["prompts"]["groups"]["behavior_batch"])
    & behavior["condition"].isin(["gold", "blue"])
]
for row in adapted.sort_values(["prompt_type", "prompt_id", "condition"]).to_dict("records"):
    print("=" * 100)
    print(row["prompt_id"], "|", row["prompt_type"], "|", row["condition"])
    print("own leak:", row["own_secret_leaked"], "| differs from base:", row["differs_from_base"])
    print("PROMPT:", row["messages"][0]["content"])
    print("OUTPUT:", row["output_text"])

batch_review_path = ensure_batch_review_template(
    paths, config, config["prompts"]["groups"]["behavior_batch"]
)
print("Published-prompt review file:", batch_review_path)


In [ ]:
APPROVE_PUBLISHED_PROMPT_GATE = False  # Change deliberately after all outputs.
BATCH_REVIEWER = ""
BATCH_REVIEW_NOTES = ""

batch_review = json.loads(batch_review_path.read_text(encoding="utf-8"))
if APPROVE_PUBLISHED_PROMPT_GATE:
    batch_review.update({
        "approved": True,
        "reviewer": BATCH_REVIEWER,
        "notes": BATCH_REVIEW_NOTES,
        "checks": {
            "all_adapter_outputs_reviewed": True,
            "standard_prompts_show_relevant_taboo_behavior": True,
            "direct_prompts_resist_literal_extraction": True,
            "own_secret_leakage_is_acceptable": True,
            "adapters_differ_meaningfully_from_base": True,
        },
    })
    batch_review_path.write_text(json.dumps(batch_review, indent=2), encoding="utf-8")

from src.behavior import require_behavior_approval
require_behavior_approval(paths, config)
print("All behavior gates passed; notebook 03 is unlocked.")
